# 🐸 Wednesday Frog – Flow Matching Training (Kaggle / Colab)

Zwei Stufen: **1. Pretrain** auf Froschfotos + Memes (lernt, wie Frösche aussehen), **2. Fine-Tune** nur auf die Memes.

**Kaggle:** Settings → Accelerator = *GPU T4 x2* oder *P100*, Internet = *On*. Unter *Add Input* zwei eigene Datasets hinzufügen:
- `wednesday-frogs` ← `data/wednesday-frogs.zip`
- `frog-photos` ← `data/pretrain/frog-photos.zip`

(Beide Zips dürfen auch zusammen in **einem** Dataset liegen – gesucht wird nach den Ordnern `wednesday-frogs/128` und `frog-photos/128`, egal wie das Dataset heißt.)

**Colab:** Runtime → GPU. Beide Zips in Google Drive (`MyDrive/`) legen.

Bei Timeout einfach alles nochmal ausführen – `train.py` setzt automatisch bei `last.pt` fort (sofern das Output-Verzeichnis erhalten bleibt, z.B. auf Drive).

In [ ]:
REPO = "https://github.com/alexanderulbrich/wednesday.git"
!git clone -q $REPO wednesday || (cd wednesday && git pull -q)
%cd wednesday
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os

def find_dataset(name):
    """Der Ordner name/128 mit den Bildern – egal wo Kaggle/Colab ihn einhängt.

    Der Ordner direkt über 128/ muss exakt `name` heißen. So klappt es mit zwei getrennten Datasets
    (/kaggle/input/frog-photos/128) genauso wie mit beiden Zips in einem gemeinsamen Dataset
    (/kaggle/input/irgendwas/frog-photos/128).
    """
    hits = sorted(glob.glob(f"{ROOT}/**/{name}/128", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{name}/128' nicht gefunden unter {ROOT} – Dataset als Input hinzugefügt? "
                                f"Inhalt prüfen mit: !find {ROOT} -maxdepth 5 -type d")
    if len(hits) > 1:
        print(f"Achtung, mehrere Treffer für {name}, nehme den ersten: {hits}")
    return hits[0]

if os.path.exists("/kaggle"):
    ROOT, OUT = "/kaggle/input", "/kaggle/working"
else:  # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, OUT = "/content/data", "/content/drive/MyDrive/wednesday-runs"   # Runs auf Drive -> überleben Timeouts
    for z in ("wednesday-frogs", "frog-photos"):
        !mkdir -p {ROOT}/{z} && unzip -qo /content/drive/MyDrive/{z}.zip -d {ROOT}/{z}

MEMES, PHOTOS = find_dataset("wednesday-frogs"), find_dataset("frog-photos")
RES = 64   # trainiert wird in 64 px, Quelle sind die schärferen 128er Bilder
for d in (MEMES, PHOTOS):
    print(d, len(os.listdir(d)), "Bilder")

## Schneller Prototyp (~30 min)
Kleineres Netz (`--base 32`, ~4,5M statt 18M Parameter) und ein Zeitbudget: `--minutes 25` misst nach ~120 Schritten das Tempo und legt die Schrittzahl so fest, dass das Training nach 25 Minuten fertig ist. Eigener Ordner `proto/`, damit nichts mit dem großen Lauf kollidiert. Danach unten direkt Frösche erzeugen.

In [ ]:
!python train.py --data {PHOTOS},{MEMES} --res {RES} --base 32 --minutes 25 --steps 100000 --lr 3e-4 --warmup 500 --batch 64 --out {OUT}/proto --sample-every 1000 --val-every 1000 --save-every 2000 --workers 2

In [ ]:
# Frösche aus dem Prototyp erzeugen + Loss-Kurven anzeigen
!python generate.py --ckpt {OUT}/proto/frog.pt --n 16 --grid --size 128 --no-open --out {OUT}/proto/generated
from IPython.display import Image, display
display(Image(f"{OUT}/proto/loss.png"))
display(Image(sorted(glob.glob(f"{OUT}/proto/generated/*.png"))[-1]))

## Stufe 1: Pretrain (Froschfotos + Memes)

In [ ]:
!python train.py --data {PHOTOS},{MEMES} --res {RES} --out {OUT}/pre --steps 40000 --batch 64 --sample-every 2000 --save-every 5000 --workers 2

## Stufe 2: Fine-Tune (nur Memes)
Kürzer und mit kleinerer Lernrate. In den Sample-Rastern schauen, ab wann die Bilder nur noch Trainingsbilder kopieren – dann lieber einen früheren Stand nehmen.

In [ ]:
!python train.py --init {OUT}/pre/last.pt --data {MEMES} --res {RES} --out {OUT}/ft --steps 10000 --lr 1e-4 --warmup 200 --batch 64 --sample-every 1000 --save-every 2000 --workers 2

## Fortschritt anschauen
Während ein Training läuft, ist das Notebook blockiert – dann `loss.png` und `samples/` rechts unter *Output* öffnen.

In [ ]:
from IPython.display import Image, display
RUN = "ft"   # oder "pre"
# Loss-Kurven (links Training, rechts Trainings- vs. Validierungsbilder; Erklärung in wednesday/monitor.py)
if os.path.exists(f"{OUT}/{RUN}/loss.png"):
    display(Image(f"{OUT}/{RUN}/loss.png"))
for s in sorted(glob.glob(f"{OUT}/{RUN}/samples/*.png"))[-3:]:
    print(s); display(Image(s))

Am Ende `frog.pt` (nur EMA-Gewichte, klein) aus dem Output herunterladen und auf dem Mac `python generate.py --ckpt frog.pt` ausführen.

In [ ]:
# Zwischenstand als frog.pt exportieren (falls ein Training noch nicht fertig ist)
import torch
RUN = "ft"   # oder "pre"
ck = torch.load(f"{OUT}/{RUN}/last.pt", map_location="cpu", weights_only=False)
torch.save({"ema": ck["ema"], "model_config": ck["model_config"], "step": ck["step"]}, f"{OUT}/{RUN}/frog.pt")
print("step", ck["step"], "->", f"{OUT}/{RUN}/frog.pt")